LAB 5

In [13]:
con.execute("""
    CREATE OR REPLACE TABLE bronze_customers AS
    SELECT DISTINCT
        customer_id, name, cpf, email, segment,
        CAST(credit_score AS INT) AS credit_score,
        CAST(created_at AS DATE) AS created_at
    FROM read_csv_auto('bigdata/raw/customers/customers_synthetic.csv')
    WHERE customer_id IS NOT NULL AND credit_score BETWEEN 300 AND 900
""")
print(con.execute("SELECT COUNT(*) FROM bronze_customers").fetchone()[0])

con.execute("""
    CREATE OR REPLACE TABLE bronze_transactions AS
    SELECT DISTINCT
        transaction_id, customer_id,
        CAST(amount AS FLOAT) AS amount,
        transaction_type, status,
        CAST(risk_score AS FLOAT) AS risk_score,
        CASE WHEN is_fraud = 'True' THEN true ELSE false END AS is_fraud,
        CAST(timestamp AS TIMESTAMP) AS ts
    FROM read_csv_auto('bigdata/raw/transactions/transactions_synthetic.csv')
    WHERE amount > 0 AND customer_id IS NOT NULL
""")
print(con.execute("SELECT MIN(amount), MAX(amount), COUNT(*) FROM bronze_transactions").fetchdf())

9993
   min(amount)   max(amount)  count_star()
0         10.0  22927.023438        100000


In [14]:
print(con.execute("""
    SELECT transaction_id, COUNT(*) c FROM bronze_transactions
    GROUP BY transaction_id HAVING c > 1
""").fetchdf())

print(con.execute("""
    SELECT is_fraud, COUNT(*) FROM bronze_transactions GROUP BY is_fraud
""").fetchdf())

Empty DataFrame
Columns: [transaction_id, c]
Index: []
   is_fraud  count_star()
0     False         98167
1      True          1833


In [15]:
con.execute("COPY bronze_customers TO 'bigdata/bronze/customers.parquet' (FORMAT PARQUET)")
con.execute("COPY bronze_transactions TO 'bigdata/bronze/transactions.parquet' (FORMAT PARQUET)")
print("Bronze salva em bigdata/bronze/")

Bronze salva em bigdata/bronze/
